# 0. Import libraries
* 코드 구동을 위한 라이브러리 호출

In [ ]:
import os
import pickle
import random
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from datetime import timedelta
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.autograd import Variable

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

print("Current directory: ", os.getcwd())

if not os.path.exists('./Results'):
    os.mkdir('./Results')
    
#### Create folders
if not os.path.exists('./Model'):
    os.mkdir('./Model')
    
#### Set seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available)
print(torch.version.cuda)
print(device)

class QuantileLoss(nn.Module):
    def __init__(self, quantile):
        super(QuantileLoss, self).__init__()
        assert 0 < quantile < 1, "Quantile should be in (0, 1) range"
        self.quantile = quantile

    def forward(self, preds, target):
        errors = target - preds
        loss = torch.max((self.quantile - 1) * errors, self.quantile * errors)
        return loss.mean()

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_ts(y_test, y_pred,
            index,  # plotting period
            show=True,
            save=False,
            target=None,
            model=None,
            fct_h=None,
            seq_len=None):

    y_test = y_test.ravel()
    y_pred = y_pred.ravel()

    y_pred[y_pred < 0] = 0

    plt.figure(figsize=(10, 4))

    # Create a DataFrame to identify continuous segments
    df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred}, index=index)

    # Find gaps in the index
    gaps = df.index.to_series().diff().dt.days.ne(1).cumsum()

    # Plot each continuous segment separately
    last_end = None
    for _, segment in df.groupby(gaps):
        if last_end is not None:
            plt.axvspan(last_end, segment.index[0], color='gray', alpha=0.3)
        plt.plot(segment.index, segment['y_test'], c='black', linewidth=1)
        plt.plot(segment.index, segment['y_pred'], c='red', linewidth=1)
        last_end = segment.index[-1]

    # Optional: scatter plot to highlight points
    plt.axhline(y=10, color='gray', linestyle='--', linewidth=2, alpha=0.5)
    plt.axhline(y=30, color='gray', linestyle='--', linewidth=2, alpha=0.5)
    plt.axhline(y=100, color='gray', linestyle='--', linewidth=2, alpha=0.5)

    plt.xlim(index[0], index[-1])
    plt.ylim(0 - y_test.max() * 0.05, max(y_test.max() * 1.05, y_pred.max() * 1.05))
    plt.yticks(size=13)
    plt.ylabel('Predicted Turbidity (NTU)', fontsize=13)
    plt.tight_layout()

    if save:
        folder_nam = "Model_evaluation"  # Modify this to your desired folder name

        import os

        if not os.path.exists(f'./Results/{folder_nam}'):
            os.mkdir(f'./Results/{folder_nam}')

        plt.savefig(f'./Results/{folder_nam}/{model}_{target}_{fct_h}_{seq_len}_temporal_variation.png', 
                    dpi=256, transparent=True)

    if show:
        plt.show()
    else:
        plt.close()
        
def calculate_score(y_test, y_pred, target):
    from sklearn.metrics import r2_score, mean_squared_error

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    accuracy, recall, _ = binning(y_test, y_pred, target)

    return np.round(rmse, 4), np.round(r2, 4), np.round(accuracy, 4), np.round(recall, 4)

def binning(y_test, y_pred, target):
    from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
    
    if target in ['Synedra', 'ToxicCyano']: 
        cri = [-np.inf, 100, 300, 1000, np.inf]
        cri2 = [-np.inf, 0.02, np.inf]
    if target in ['2MIB', 'Geosmin']: 
        cri = [-np.inf, 0.01, 0.02, 0.05, np.inf]
        cri2 = [-np.inf, 0.02, np.inf]

    if "WTemp" in target:    
        cri = [-np.inf, 10, 20, 30, np.inf]  # Criteria for binning
        cri2 = [-np.inf, 30, np.inf]  # Criteria for binning    

    if "NTU" in target:
        cri = [-np.inf, 10, 30, 100, np.inf]  # Criteria for binning
        cri2 = [-np.inf, 30, np.inf]  # Criteria for binning

    if "pH" in target:
        cri = [-np.inf, 10, 30, 100, np.inf]  # Criteria for binning
        cri2 = [-np.inf, 30, np.inf]  # Criteria for binning

    y_true_binned = pd.cut(y_test, bins=cri, labels=[0,1,2,3])
    y_pred_binned = pd.cut(y_pred, bins=cri, labels=[0,1,2,3])
    y_true_binned2 = pd.cut(y_test, bins=cri2, labels=[0,1])
    y_pred_binned2 = pd.cut(y_pred, bins=cri2, labels=[0,1])
    
    return accuracy_score(y_true_binned, y_pred_binned), recall_score(y_true_binned2, y_pred_binned2), confusion_matrix(y_true_binned, y_pred_binned, labels=[0, 1, 2, 3])

## Function for save R2, RMSE, accuracy and recall
def performance_tab_mod(y_te, y_pred_te,
                    fct,
                    model_nam,
                    info = None, ###
                    target = "NTU"):
    
    perf_save_temp = pd.DataFrame(columns = ['rmse', 'r2', 'accuracy', 'recall', 'fct', 'site', 'model', 'info'])
    
    perf_save_temp.loc[0, ['rmse', 'r2', 'accuracy', 'recall']] = calculate_score(y_te.ravel(), y_pred_te.ravel(), target)

    perf_save_temp.loc[0, 'data'] = 'te'

    perf_save_temp.loc[:, "fct"] = fct
    perf_save_temp.loc[:, "site"] = target
    perf_save_temp.loc[:, "model"] = model_nam
    perf_save_temp.loc[:, "info"] = info
    
    return perf_save_temp

def confusion_tab_pred(y_, y_pred_,
                    fct,
                    model_nam,
                    info = None, ###
                    target = "NTU"):
    
    columns = [''] + [f'plot1_{i}_{j}' for i in range(4) for j in range(4)] + ['accuracy'] + ['recall'] + ['data']
    conf_save_temp = pd.DataFrame(columns=columns)
    
    accuracy, recall, conf = binning(y_.ravel(), y_pred_.ravel(), target)
    add_metrics_to_df(conf_save_temp, 0, pd.DataFrame(conf), accuracy, recall, 'tr')

    conf_save_temp.loc[:, "fct"] = fct
    conf_save_temp.loc[:, "site"] = target
    conf_save_temp.loc[:, "model"] = model_nam
    conf_save_temp.loc[:, "Info"] = info

    return conf_save_temp

def confusion_matrix_file(df): ### TODO modifying code for weekly models
    n = len(df)

    new_df = pd.DataFrame()
    cols = ['10 to 30', '30 to 100', '100 이상', 'Accuracy', 'Recall', '']

    for i in range(n):
        temp = df.iloc[i:i + 1].copy()
        temp.iloc[0, 0] = '0 to 10'
        new_df = pd.concat([new_df, temp], ignore_index=True)

        for j, value in enumerate(cols):
            new_row = [''] * len(df.columns)
            new_row[0] = value
            new_df = pd.concat([new_df, pd.DataFrame([new_row], columns=df.columns)], ignore_index=True)
        
        new_df.iloc[i * (len(cols) + 1), 5:10] = df.iloc[i, 19:24].values
        new_df.iloc[i * (len(cols) + 1) + 1, 1:5] = df.iloc[i, 5:9].values
        new_df.iloc[i * (len(cols) + 1) + 2, 1:5] = df.iloc[i, 9:13].values
        new_df.iloc[i * (len(cols) + 1) + 3, 1:5] = df.iloc[i, 13:17].values
        new_df.iloc[i * (len(cols) + 1) + 4, 1] = df.iloc[i, 17]
        new_df.iloc[i * (len(cols) + 1) + 5, 1] = df.iloc[i, 18]
        new_df.drop(new_df.columns[10:24], axis=1, inplace=True)
        
    new_df.columns = ['','0 to 10', '10 to 30','30 to 100','100 이상', 'data', 'fct', 'site', 'model', 'Info']
    
    return new_df 

def add_metrics_to_df(df, row_index, plot1, accuracy, recall, info):
    for i in range(4):
        for j in range(4):
            df.loc[row_index, f'plot1_{i}_{j}'] = plot1.iloc[i, j]
    df.loc[row_index, 'accuracy'] = accuracy
    df.loc[row_index, 'recall'] = recall
    df.loc[row_index, 'data'] = info
    
def is_defined(variable_name): ## Utility function
    try:
        eval(variable_name)
        return True
    except NameError:
        return False
    
def create_sequences(data, target, knw_data, known,
                     seq_length, forecast_horizon, horizon_length, work_hour):
    xs = []
    ys = []
    knws = []
    
    for t in data[data.index.hour == work_hour].index:
        end_time = t
        start_time = end_time - pd.Timedelta(hours = seq_length - 1)
        
        x = data.loc[start_time:end_time, predictors].values
        if x.shape[0] == seq_length:
            xs.append(x)
            
            # x_ = data.loc[start_time:end_time, predictors]
            # if not is_defined('xs'):
            #     xs = np.expand_dims(x, axis = 0)
            # else:
            #     xs = np.concatenate((xs, np.expand_dims(x, axis = 0)))
    
    xs = np.array(xs)
    
    #### For target sequence
    sel_time = data[data.index.hour == work_hour][-xs.shape[0]:].index.floor('D')
    for t in sel_time:
        start_time = t + pd.Timedelta(days = forecast_horizon) 
        end_time = start_time + pd.Timedelta(days = horizon_length)

        y = data.loc[start_time:end_time][:-1]
        
        if y.shape[0] == horizon_length * 24:
            y.loc[:, 'date'] = y.index.date
            y_ = y.groupby('date')[target].mean().values
            ys.append(y_)
            
            knw = knw_data.loc[pd.to_datetime(knw_data['Date']) == t, known].values
            knws.append(knw)
            
            # if not is_defined('ys'):
            #     ys = np.expand_dims(y_, axis = 0)
            # else:        
            #     ys = np.concatenate((ys, np.expand_dims(y_, axis = 0)))
            
    ys = np.array(ys)
    knws = np.array(knws)#/100
    xs = xs[:len(xs)-forecast_horizon-horizon_length, :, :]
    
    return xs, ys, knws

def create_sequences_forecasts(test_data_, test_knw, known_vars, predictors, seq_len, fct_h, wrk_t):
    sequences = []
    knowns = []
    
    for current_time in test_data_[test_data_.index.hour == wrk_t].index:
        start_time = current_time - pd.Timedelta(hours=seq_len - 1)
        sequence_data = test_data_.loc[start_time:current_time, predictors].values
        if sequence_data.shape[0] == seq_len:
            sequences.append(sequence_data)
    
    sequences = np.array(sequences)
    aligned_dates = test_data_[test_data_.index.hour == wrk_t][-sequences.shape[0]:].index.floor('D')
    
    for current_date in aligned_dates:
        forecast_time = current_date + pd.Timedelta(days=fct_h)
        known_data = test_knw.loc[pd.to_datetime(test_knw['Date']) == current_date, known_vars].values
        knowns.append(known_data)
    
    knowns = np.array(knowns)
    
    return sequences, knowns

def create_sequences_forecasts(test_data_, test_knw, known_vars, predictors, seq_len, fct_h, wrk_t):
    sequences = []
    knowns = []
    
    for current_time in test_data_[test_data_.index.hour == wrk_t].index:
        start_time = current_time - pd.Timedelta(hours=seq_len - 1)
        sequence_data = test_data_.loc[start_time:current_time, predictors].values
        if sequence_data.shape[0] == seq_len:
            sequences.append(sequence_data)
    
    sequences = np.array(sequences)
    aligned_dates = test_data_[test_data_.index.hour == wrk_t][-sequences.shape[0]:].index.floor('D')
    
    for current_date in aligned_dates:
        forecast_time = current_date + pd.Timedelta(days=fct_h)
        known_data = test_knw.loc[pd.to_datetime(test_knw['Date']) == current_date, known_vars].values
        knowns.append(known_data)
    
    knowns = np.array(knowns)
    
    return sequences, knowns

class TimeSeriesDataset_pred(Dataset):
    def __init__(self, x, knw, device):
        self.x = torch.tensor(x, dtype=torch.float32).to(device)
        self.knw = torch.tensor(knw, dtype=torch.float32).to(device)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
        return self.x[index], self.knw[index]
    
def Forecasts(test_data_, y_pred_te, target, wrk_t, seq_len):
    selected_dates = test_data_[test_data_.index.hour == wrk_t][int(np.round(seq_len / 24)):].index.floor('D')
    test_data_.loc[:, 'date'] = test_data_.index.date
    daily_mean = test_data_.groupby('date')[target].mean().values
    df_daily_mean = pd.DataFrame(daily_mean, index=test_data_.index.floor('D').unique(), columns=[target])
    df_daily_mean = df_daily_mean.loc[selected_dates, :]
    combined_df = pd.concat([df_daily_mean, pd.DataFrame(y_pred_te, index=selected_dates)], axis=1)
    last_row = combined_df.iloc[-1, :]
    for i in range(7):
        combined_df.iloc[:, i + 1] = combined_df.iloc[:, i + 1].shift(i + 1)
    last_date = combined_df.index[-1] + pd.Timedelta(days=1)
    forecast_end_date = last_date + pd.Timedelta(days=6)
    forecast_df = pd.DataFrame(np.array(last_row[1:]).reshape(-1, 1), columns=['forecasts'], index=pd.date_range(last_date, forecast_end_date, freq="D"))
    return forecast_df, combined_df, last_row

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def rrmse(targets, predictions):
    return 100 * np.sqrt(((predictions - targets) ** 2).mean()) / np.mean(targets)

# Define NNSE function
def nnse(observed, modeled):
    if len(observed) != len(modeled):
        raise ValueError("Observed and modeled arrays must have the same length.")
    mean_observed = np.mean(observed)
    numerator = np.sum((observed - modeled) ** 2)
    denominator = np.sum((observed - mean_observed) ** 2)
    nse = 1.0 - (numerator / denominator)
    nnse = 1.0 / (2.0 - nse)
    return nnse

# Define NSE function
def nse(observed, modeled):
    if len(observed) != len(modeled):
        raise ValueError("Observed and modeled arrays must have the same length.")
    mean_observed = np.mean(observed)
    numerator = np.sum((observed - modeled) ** 2)
    denominator = np.sum((observed - mean_observed) ** 2)
    nse = 1.0 - (numerator / denominator)
    return nse


def quantileloss(p_dat, min_, max_, quantile = 0.8):
    total_quantile_loss = 0.0
    total_mse_loss = 0.0
    min = min_[0]
    max = max_[0]
    p_dat_ = (p_dat - min)/(max - min)

    for i in range(7):
        errors = p_dat_[:, 0] - p_dat_[:, i + 1]
        loss = np.maximum((quantile - 1) * errors, quantile * errors)
        #print(np.mean(loss))
        total_quantile_loss += np.mean(loss)
    
        mse_loss = errors ** 2
        total_mse_loss += np.mean(mse_loss)
    
    # 평균 Quantile Loss와 평균 MSE Loss 반환
    avg_quantile_loss = total_quantile_loss / 7
    avg_mse_loss = total_mse_loss / 7

    return avg_quantile_loss, avg_mse_loss


def loss_summertime(p_dat, min_, max_, i):
    from sklearn.metrics import recall_score, mean_squared_error, mean_absolute_error
    import numpy as np
    p_dat_sum = p_dat.loc[
        ((p_dat.index >= '2023-07-01') & (p_dat.index < '2023-10-01')) |
        ((p_dat.index >= '2024-07-01') & (p_dat.index < '2024-10-01'))
    ]
    min = min_[0]
    max = max_[0]

    p_dat_sum_ = (p_dat_sum - min)/(max - min)
    p_dat_ = (p_dat - min)/(max - min)
    p_dat__ = np.array(p_dat_.dropna())
    p_dat___ = p_dat.dropna() 
    p_dat____ = (p_dat___ - min)/(max - min)


    rrmse_ = rrmse(p_dat__[:,0], p_dat__[:,i])
    nnse_ = nnse(p_dat__[:,0], p_dat__[:,i])
    nse_ = nse(p_dat__[:,0], p_dat__[:,i])    

    rmse = np.sqrt(mean_squared_error(p_dat___[target], p_dat___[i]))
    mse = mean_squared_error(p_dat___[target], p_dat___[i])
    mae = mean_absolute_error(p_dat___[target], p_dat___[i])

    rmse_ = np.sqrt(mean_squared_error(p_dat____[target], p_dat____[i]))
    mse_ = mean_squared_error(p_dat____[target], p_dat____[i])
    mae_ = mean_absolute_error(p_dat____[target], p_dat____[i])

    rmse_sum = np.sqrt(mean_squared_error(p_dat_sum_[target], p_dat_sum_[i]))
    mse_sum = mean_squared_error(p_dat_sum_[target], p_dat_sum_[i])
    mae_sum = mean_absolute_error(p_dat_sum_[target], p_dat_sum_[i])

    return rrmse_, nnse_, nse_, rmse, mse, mae, rmse_, mse_, mae_, rmse_sum, mse_sum, mae_sum

In [ ]:
# Functions for forecasts
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import os
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, confusion_matrix, accuracy_score, recall_score

### Custom model
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(Encoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.bn = nn.BatchNorm1d(hidden_dim * 2)
        
    def forward(self, input): ### input == xs
        # x shape: (batch_size, seq_len, input_dim)
        output, (hidden, cell) = self.lstm(input)
        
        # Apply batch normalization
        output = output.transpose(1, 2)  # (batch_size, hidden_dim * 2, seq_len)
        output = self.bn(output)
        output = output.transpose(1, 2)  # (batch_size, seq_len, hidden_dim * 2)
        
        return output, hidden, cell

class Decoder(nn.Module):
    def __init__(self, hidden_dim, num_layers, known_feature_dim):
        super(Decoder, self).__init__()
        
        ### known_feature_dim = 1 (Weather forecasts)
        self.hidden_dim = hidden_dim * 2
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(self.hidden_dim + known_feature_dim, int(self.hidden_dim/2), int(self.num_layers*2), batch_first=True, bidirectional=False)
        
    def forward(self, input, known_features, hidden, cell): # input == xs
        # x shape: (batch_size, 1, input_dim)
        # known_features shape: (batch_size, 1, known_feature_dim)
        combined_input = torch.cat((input, known_features), dim=2)
        
        output, (hidden, cell) = self.lstm(combined_input, (hidden, cell))
        
        return output, hidden, cell

class ED_BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, known_feature_dim, forecast_horizon):
        super(ED_BiLSTM, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, num_layers)
        self.decoder = Decoder(hidden_dim, num_layers, known_feature_dim)
        self.forecast_horizon = forecast_horizon
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, known_features, target=None, teacher_forcing_ratio=0.2):
        # input shape: (batch_size, input_seq_len = 72, input_dim)
        # known_features shape: (batch_size, output_seq_len = 7, known_feature_dim = 1)
        # output shape: (batch_size, output_seq_len = 7, output_dim = 1)
        
        out_seq_len = self.forecast_horizon
        encoder_output, hidden, cell = self.encoder(input)
        
        # Use the last output from the encoder as the first input to the decoder
        outputs = []
        decoder_input = encoder_output[:, -1:, :]

        for t in range(out_seq_len):
            decoder_output, hidden, cell = self.decoder(decoder_input, known_features[:, t:t+1, :], hidden, cell)
            
            prediction = self.fc(decoder_output.squeeze(1))
                
            outputs.append(prediction)
                
        outputs = torch.cat(outputs, dim = 1)
        return outputs

In [ ]:
## Directory
os.chdir('/home/uoswem7/Dogeon/논문용_모델/Turbidity')

# LTF-EDLSTM

In [ ]:
dep_vars = ['PD1.NTU', 'PD2.NTU', 'PD3.NTU']
predictors = ['CJdam.Prec', 'PDdam.Prec', 'CJdam.DIS', 'CPdam.DIS', 'PDdam.DIS',
            #   'YP.NTU', 'GP.NTU', 
              'CP.Prec', 'GJ.Prec', 'YP.Prec', 'CJdam.NTU', 'SYdam.NTU'
            #   'YP.WTemp', 'GP.WTemp'
              ]
known_vars= []

targets = ['PD1.NTU', 'PD2.NTU', 'PD3.NTU']
Variables = dep_vars + predictors

file_nams = ['Turb_NoAR_future', 'Turb_AR_future', 'Turb_NoAR_Nofuture', 'Turb_AR_Nofuture']
loss_nams = ['huber_loss', 'mse_loss', 'mae_loss']

In [ ]:
import warnings
warnings.filterwarnings('ignore')
sepa = "2022-12-31 23:00"
model_nam = "BiLSTM_ED"

seq_len = 36
fct_h = 1
h_len = 7
wrk_t = 9
performance_total_all = pd.DataFrame()
for file_nam in file_nams:
    for loss_nam in loss_nams:
        if not os.path.exists('./Results/%s'%(file_nam)):
            os.mkdir('./Results/%s'%(file_nam))    

        if "Nofuture" in file_nam:
            known_vars= []
            class Decoder(nn.Module):
                def __init__(self, hidden_dim, num_layers, known_feature_dim):
                    super(Decoder, self).__init__()
                    
                    known_feature_dim = 0 #(Weather forecasts)
                    self.hidden_dim = hidden_dim * 2
                    self.num_layers = num_layers
                    
                    self.lstm = nn.LSTM(self.hidden_dim + known_feature_dim, int(self.hidden_dim/2), int(self.num_layers*2), batch_first=True, bidirectional=False)
                    
                def forward(self, input, known_features, hidden, cell): # input == xs
                    # x shape: (batch_size, 1, input_dim)
                    # known_features shape: (batch_size, 1, known_feature_dim)
                    combined_input = torch.cat((input, known_features), dim=2)
                    
                    output, (hidden, cell) = self.lstm(combined_input, (hidden, cell))
                    
                    return output, hidden, cell

        else:
            known_vars= ['RainProb']
            class Decoder(nn.Module):
                def __init__(self, hidden_dim, num_layers, known_feature_dim):
                    super(Decoder, self).__init__()
                    
                    known_feature_dim = 1 #(Weather forecasts)
                    self.hidden_dim = hidden_dim * 2
                    self.num_layers = num_layers
                    
                    self.lstm = nn.LSTM(self.hidden_dim + known_feature_dim, int(self.hidden_dim/2), int(self.num_layers*2), batch_first=True, bidirectional=False)
                    
                def forward(self, input, known_features, hidden, cell): # input == xs
                    # x shape: (batch_size, 1, input_dim)
                    # known_features shape: (batch_size, 1, known_feature_dim)
                    combined_input = torch.cat((input, known_features), dim=2)
                    
                    output, (hidden, cell) = self.lstm(combined_input, (hidden, cell))
                    
                    return output, hidden, cell

        performance_table = pd.DataFrame()
        for target in targets:
            if loss_nam == 'huber_loss':
                if "_NoAR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 7, 64, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 7, 32, 0.005)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 7, 96, 0.005)]

                elif "_AR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(32, 7, 128, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 7, 32, 0.005)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 5, 64, 0.0001)]

                elif "_NoAR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 32, 0.0001)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 5, 128, 0.0001)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 5, 64, 0.005)]

                elif "_AR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 7, 32, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(64, 7, 64, 0.005)] 
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(32, 3, 96, 0.005)]

            elif loss_nam == 'mse_loss':
                if "_NoAR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 64, 0.0005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 5, 128, 0.005)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]

                elif "_AR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(32, 5, 64, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 5, 64, 0.005)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 5, 64, 0.005)]

                elif "_NoAR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 64, 0.0005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(32, 3, 128, 0.0001)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 3, 64, 0.005)]

                elif "_AR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(32, 5, 64, 0.005)] 
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(32, 3, 128, 0.005)]
        
            elif loss_nam == 'mae_loss':
                if "_NoAR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(32, 5, 64, 0.005)]

                elif "_AR" in file_nam and "_Nofuture" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]

                elif "_NoAR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(32, 5, 128, 0.005)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 3, 64, 0.0001)]
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0001)]

                elif "_AR" in file_nam and "_future" in file_nam:
                    if target == 'PD1.NTU':
                        hyperparameter_combinations = [(16, 5, 128, 0.0001)]
                    elif target == 'PD2.NTU':
                        hyperparameter_combinations = [(16, 5, 128, 0.0001)] 
                    elif target == 'PD3.NTU':
                        hyperparameter_combinations = [(16, 3, 128, 0.0005)]

            Model_future = pd.read_csv(f'./Data/Model_known_.csv')#, index_col = 0)
            test_data_imp = pd.read_csv(f"./Data/test_data_sensor_.csv", index_col=0, encoding = "cp949")
            test_data_imp.index = pd.to_datetime(test_data_imp.index)
            test_data_imp = test_data_imp[test_data_imp.index >= pd.to_datetime(sepa)]
            #Model_data_hour_imp = pd.read_csv(f'./Data/test_data_sensor_.csv', index_col = 0)[Variables] # with imputation - UKF

            desired_datetime = test_data_imp.index[-1].normalize() + pd.Timedelta(hours=wrk_t)
            if desired_datetime > test_data_imp.index[-1]:
                desired_datetime -= pd.Timedelta(days=1)
            
            test_data_imp = test_data_imp.loc[:desired_datetime]

            #Model_data_hour_imp.index = pd.to_datetime(Model_data_hour_imp.index).round('S')
            #Model_data_hour_imp.index = pd.to_datetime(Model_data_hour_imp.index).floor('T')
            #Model_data_hour_imp = Model_data_hour_imp[Model_data_hour_imp.index >= test_data_imp.index[0]]

            # ###########
            # test_data_imp = test_data_imp[test_data_imp.index<='2024-07-19 10:00']

            Model_future.index = pd.to_datetime(Model_future['Date'])

            test_knw = Model_future[Model_future.index >= pd.to_datetime(sepa)]

            ##### For create dataset
            class TimeSeriesDataset_pred(Dataset):
                def __init__(self, x, knw, device):
                    self.x = torch.tensor(x, dtype=torch.float32).to(device)
                    self.knw = torch.tensor(knw, dtype=torch.float32).to(device)
                    
                def __len__(self):
                    return len(self.x)
                
                def __getitem__(self, index):
                    return self.x[index], self.knw[index]
                

            confusion_table = pd.DataFrame()
            # target = 'PD2.NTU' ## 
            test_data_ = test_data_imp.loc[:, [target] + predictors]

            #model_data_ = Model_data_hour_imp.loc[:, [target] + predictors]

            te_x_ = []
            mod_x_ = []
            te_knw_ = []
            if "NoAR" in file_nam:
                for t in test_data_[test_data_.index.hour == wrk_t].index:
                    end_time = t
                    start_time = end_time - pd.Timedelta(hours = seq_len - 1)
                    
                    x = test_data_.loc[start_time:end_time, predictors].values
                    if x.shape[0] == seq_len:
                        te_x_.append(x)

            else:   
                for t in test_data_[test_data_.index.hour == wrk_t].index:
                    end_time = t
                    start_time = end_time - pd.Timedelta(hours = seq_len - 1)
                    
                    x = test_data_.loc[start_time:end_time, [target] + predictors].values
                    if x.shape[0] == seq_len:
                        te_x_.append(x)

            te_x_ = np.array(te_x_)
            #mod_x_ = np.array(mod_x_)

            min_, max_ = pickle.load(open(f'./Model/Turb_LTFEDLSTM/{file_nam}/{target}_Minmax.pkl', 'rb'))
            sel_time = test_data_[test_data_.index.hour == wrk_t][-te_x_.shape[0]:].index.floor('D')
            for t in sel_time[:]:
                start_time = t + pd.Timedelta(days = fct_h) 

                knw = test_knw.loc[pd.to_datetime(test_knw.index) == t, known_vars].values
                te_knw_.append(knw)
                    
            te_knw_ = np.array(te_knw_)/100

            #te_x_ = te_x_[:-1, :, :]

            # print(te_x_.shape, te_knw_.shape)
            te_x__ = (te_x_ - min_[1:])/(max_[1:] - min_[1:])
            #mod_x__ = (mod_x_ - min_[1:])/(max_[1:] - min_[1:])

            pred_dataset = TimeSeriesDataset_pred(te_x__, te_knw_, device)
            #pred_dataset_mod = TimeSeriesDataset_pred(mod_x__, te_knw_, device)

            test_data_.loc[:, 'date'] = test_data_.index.date
            test_data__ = test_data_.groupby('date')[target].mean().values
            y_te = pd.DataFrame(test_data__, index = test_data_.index.floor('D').unique(), columns = [target])
            y_te = y_te.loc[sel_time, :]
            for hidden_dim, num_layers, bs, lr in hyperparameter_combinations:
                print(file_nam)
                print(loss_nam)
                print(target)
                print('hidden_dim:', hidden_dim)
                print('num_layer:', num_layers)
                print('batch_size:', bs)
                print('Learning_rate:', lr)

                model = torch.load('./Model/Turb_LTFEDLSTM/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam ,loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)).to(device)
                #model = torch.load('./Model/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam ,model_nam, target, bs, hidden_dim, num_layers, lr)).to(device)
                #model = torch.load('./Model/%s/%s_1_%s_%s.pt'%(file_nam ,model_nam, target, info)).to(device)
                # sel_time -> LABEL
                
                if os.path.exists('./Model/Turb_LTFEDLSTM/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt' % (
                    file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr
                )):
                    model_size_mb = os.path.getsize('./Model/Turb_LTFEDLSTM/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt' % (
                    file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr
                )) / (1024 * 1024)
                    print(f"📦 Model size: {model_size_mb:.2f} MB → {os.path.basename('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr))}")
                else:
                    print(f"⚠️ There is no model: {'./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)}")
                                # sel_time -> LABEL``

                model.eval()
                y_pred_te = model(pred_dataset.x, pred_dataset.knw)
                y_pred_te = y_pred_te.detach().cpu().numpy() * (max_[0] - min_[0]) + min_[0]

                #y_pred_mod = model(pred_dataset_mod.x, pred_dataset_mod.knw)
                #y_pred_mod = y_pred_mod.detach().cpu().numpy() * (max_[0] - min_[0]) + min_[0]

                p_dat = pd.concat([y_te, pd.DataFrame(y_pred_te, index = sel_time)], axis = 1)
                fcts = p_dat.iloc[-2, :]

                #p_dat_mod = pd.concat([y_te, pd.DataFrame(y_pred_mod, index = sel_time)], axis = 1)
                #fcts_mod = p_dat.iloc[-2, :]

                for i in range(7):
                    p_dat.iloc[:, i+1] = p_dat.iloc[:, i+1].shift(i+1)
                    #p_dat_mod.iloc[:, i+1] = p_dat_mod.iloc[:, i+1].shift(i+1)

                p_dat['mean'] = p_dat.iloc[:, 1:7].mean(axis = 1)
                p_dat['std'] = p_dat.iloc[:, 1:7].std(axis = 1)

                p_dat_ = p_dat.dropna()
                p_dat.to_csv(f"./Results/{file_nam}/predictions_{target}.csv")

                start_time = p_dat.index[-1]
                end_time = start_time + pd.Timedelta(days = 7)

                f_dat = pd.DataFrame(np.array(fcts).reshape(-1, 1), columns = ['forecasts'],
                            index = pd.date_range(start_time, end_time, freq = "D"))
                #f_dat_mod = pd.DataFrame(np.array(fcts_mod).reshape(-1, 1), columns = ['forecasts'],
                #            index = pd.date_range(start_time, end_time, freq = "D"))
                
                f_dat.to_csv(f"./Results/{file_nam}/forcasts_{target}.csv")

                ## model = torch.load(###)

                p_dat_ = np.array(p_dat.dropna())
                #p_dat_mod_ = np.array(p_dat_mod.dropna())

                if not os.path.exists('./Results/%s'%(file_nam)):
                    os.mkdir('./Results/%s'%(file_nam))

                for i in range(7):
                    #print(i+1 ,'일 후 예측결과')
                    perf_save_temp = performance_tab_mod(p_dat_[:, 0], p_dat_[:,i+1],
                                                    i+1,
                                                    "(" + file_nam.split('_')[1] + "," +file_nam.split('_')[2] +")",
                                                    info = seq_len, 
                                                    target = target)
                    perf_save_temp['hid'] = hidden_dim
                    perf_save_temp['num_layers'] = num_layers
                    perf_save_temp['batch_size'] = bs
                    perf_save_temp['learning rate'] = lr
                    perf_save_temp['file_nam'] = file_nam
                    perf_save_temp['loss'] = loss_nam
                    rrmse_, nnse_, nse_, rmse, mse, mae, rmse_, mse_, mae_, rmse_sum, mse_sum, mae_sum = loss_summertime(p_dat, min_, max_, i)
                    

                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rrmse'] = rrmse_
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nnse'] = nnse_
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nse'] = nse_        
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse'] = mse
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae'] = mae
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_'] = rmse_
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_'] = mse_
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_'] = mae_
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_sum'] = rmse_sum
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_sum'] = mse_sum
                    perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_sum'] = mae_sum

                #     confusion_temp = confusion_tab_pred(p_dat_[:, 0], p_dat_[:,i+1],
                #                                     i+1,
                #                                     model_nam,
                #                                     info = seq_len, 
                #                                     target = target)
                    #print(i+1, '일 후 Recall :', perf_save_temp['recall'][0])
                    performance_table = pd.concat([performance_table, perf_save_temp], axis = 0)
                    performance_table['model'] = 'ED-LSTM(%s,%s)'%(file_nam.split('_')[1], file_nam.split('_')[2])
                    performance_table['target'] = 'NTU'
                    performance_table['site'] = performance_table['site'].apply(lambda x: x.split('.')[0])

                #     confusion_table = pd.concat([confusion_table, confusion_temp], axis = 0)
                    from sklearn.metrics import recall_score
                    p_dat_sum = p_dat.loc[
                        ((p_dat.index >= '2023-07-01') & (p_dat.index <= '2023-10-01')) |
                        ((p_dat.index >= '2024-07-01') & (p_dat.index <= '2024-10-01'))
                    ]

                    thresholds = [30]
                    y_true_combined = []
                    y_pred_combined = []

                    for threshold in thresholds:
                        # 임계값을 기준으로 이진 분류 (실측값과 예측값 모두 threshold를 넘으면 1, 아니면 0)
                        y_true = (p_dat_sum[target] >= threshold).astype(int)
                        y_pred = (p_dat_sum[i] >= threshold).astype(int)
                        # 리스트에 이진 결과 추가
                        y_true_combined.extend(y_true)
                        y_pred_combined.extend(y_pred)

                    recall = recall_score(y_true_combined, y_pred_combined)
                    
                    
                    if i in [0,6]:
                        #print(i+1, '일 후 Recall :', recall)
                        print(i+1, '일 후 R2 :', perf_save_temp['r2'].astype(float).values[0])
                        plt.figure(figsize=(10, 4))
                        #plt.axhline(y=30, color='gray', linestyle='--', linewidth=1, alpha=0.5)
                        #plt.axhline(y=100, color='gray', linestyle='--', linewidth=1, alpha=0.5)

                        plt.plot(p_dat[target], c = 'blue', linewidth = 1)
                        plt.plot(p_dat.iloc[:, i+1], c = 'red', linewidth = 1)
                        #plt.plot(p_dat_mod.iloc[:, i+1], c = 'red', linewidth = 1)
                        
                        #plt.title(target)
                        plt.xlim(p_dat.index[0], p_dat.index[-1])
                        plt.ylim((0, 400))
                        plt.yticks(size=10)
                        plt.ylabel('Predicted Turbidity (NTU)', fontsize=10)
                        plt.tight_layout()
                        plt.show()
                            
                        #plot_one_to_one_custom(p_dat[target], p_dat.iloc[:, i+1])

                print("")

        performance_total_all = pd.concat([performance_total_all, performance_table], axis=0)


    performance_total_all = performance_total_all[['site', 'target', 'model', 'loss', 'fct', 'hid', 'num_layers', 'batch_size',
            'learning rate', 'rmse', 'r2', 'accuracy', 'recall']]

# Comparison model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take the last time step's output
        return out
##### GRU
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

##### Bi-LSTM
class BiLSTM(nn.Module):
    torch.manual_seed(42)
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(BiLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

## LSTM, BILSTM, GRU

In [ ]:
dep_vars = ['PD1.NTU', 'PD2.NTU', 'PD3.NTU', 'GJ.NTU']
predictors = ['CJdam.Prec', 'PDdam.Prec', 'CJdam.DIS', 'CPdam.DIS', 'PDdam.DIS',
              'CP.Prec', 'GJ.Prec', 'YP.Prec' ,'CJdam.NTU', 'SYdam.NTU'
              ]
known_vars= []
Variables = dep_vars + predictors

sepa = "2023-01-01"
file_nam = 'Turb_Singlemodel'
loss_nam = 'huber_loss'

model_list = ['LSTM', 'GRU', 'BiLSTM']

targets = ['PD1.NTU','PD2.NTU' ,'PD3.NTU']

In [ ]:
import warnings
warnings.filterwarnings('ignore')
seq_len = 36
fct_h = 1
h_len = 7
info = '20240819'
#### Perform imputation
from SaaS_WQ.preprocessing import imputation_KF
from SaaS_WQ.plot import matrix

performance_table = pd.DataFrame()
confusion_table = pd.DataFrame()


for model_nam in model_list:
    p_dat_total = pd.DataFrame()
    f_dat_total = pd.DataFrame()
    for target in targets:

        if model_nam == 'LSTM':
            if target == 'PD1.NTU':
                hyperparameter_combinations = [(16, 3, 128, 0.0001)]
            elif target == 'PD2.NTU':
                hyperparameter_combinations = [(16, 3, 128, 0.0001)]
            elif target == 'PD3.NTU':
                hyperparameter_combinations = [(16, 3, 128, 0.0005)]

        elif model_nam == 'GRU':
            if target == 'PD1.NTU':
                hyperparameter_combinations = [(16, 5, 32, 0.0001)]
            elif target == 'PD2.NTU':
                hyperparameter_combinations = [(16, 7, 32, 0.005)]
            elif target == 'PD3.NTU':
                hyperparameter_combinations = [(16, 7, 128, 0.0001)]

        elif model_nam == 'BiLSTM':
            if target == 'PD1.NTU':
                hyperparameter_combinations = [(64, 7, 64, 0.005)]
            elif target == 'PD2.NTU':
                hyperparameter_combinations = [(16, 3, 128, 0.0001)]
            elif target == 'PD3.NTU':
                hyperparameter_combinations = [(64, 5, 64, 0.0005)]


        print('Model_name:', model_nam)
        if not os.path.exists('./Results/%s'%(file_nam)):
            os.mkdir('./Results/%s'%(file_nam))

        Model_future = pd.read_csv(f'./Data/Model_known_.csv')
        test_data_imp = pd.read_csv(f"./Data/test_data_sensor_.csv", index_col=0, encoding = "cp949")[Variables]
        test_data_imp.index = pd.to_datetime(test_data_imp.index)
        test_data_imp = test_data_imp[test_data_imp.index >= sepa]
        test_data_imp = test_data_imp[~test_data_imp.index.duplicated(keep='first')]

        # ###########
        # test_data_imp = test_data_imp[test_data_imp.index<='2024-07-19 10:00']

        Model_future['Date'] = pd.to_datetime(Model_future['Date'])
        #Model_future['Date'] = Model_future['Date'] + pd.Timedelta(days = 2) 

        test_knw = Model_future[Model_future['Date'] >= pd.to_datetime(sepa)]

        ##### For create dataset
        class TimeSeriesDataset_pred(Dataset):
            def __init__(self, x, knw, device):
                self.x = torch.tensor(x, dtype=torch.float32).to(device)
                self.knw = torch.tensor(knw, dtype=torch.float32).to(device)
                
            def __len__(self):
                return len(self.x)
            
            def __getitem__(self, index):
                return self.x[index], self.knw[index]
            



        wrk_t = 9
        test_data_ = test_data_imp.loc[:, [target] + predictors]

        te_x_ = []
        te_knw_ = []

        for t in test_data_[test_data_.index.hour == wrk_t].index:
            end_time = t
            start_time = end_time - pd.Timedelta(hours = seq_len - 1)
            
            x = test_data_.loc[start_time:end_time, predictors].values
            if x.shape[0] == seq_len:
                te_x_.append(x)
            
        te_x_ = np.array(te_x_)

        min_, max_ = pickle.load(open(f'./Model/{file_nam}/{target}_Minmax.pkl', 'rb'))

        #### For target sequence
        sel_time = test_data_[test_data_.index.hour == wrk_t][-te_x_.shape[0]:].index.floor('D')
        for t in sel_time[:-1]:
            start_time = t + pd.Timedelta(days = fct_h) 

            knw = test_knw.loc[pd.to_datetime(test_knw['Date']) == t, known_vars].values
            te_knw_.append(knw)
                
        te_knw_ = np.array(te_knw_)/100

        te_x__ = (te_x_ - min_[1:])/(max_[1:] - min_[1:])

        pred_dataset = TimeSeriesDataset_pred(te_x__, te_knw_, device)

        test_data_.loc[:, 'date'] = test_data_.index.date
        test_data__ = test_data_.groupby('date')[target].mean().values
        y_te = pd.DataFrame(test_data__, index = test_data_.index.floor('D').unique(), columns = [target])
        y_te = y_te.loc[sel_time, :]

        for hidden_dim, num_layers, bs, lr in hyperparameter_combinations:
            print(model_nam)
            print(target)
            print('hidden_dim:', hidden_dim)
            print('num_layer:', num_layers)
            print('batch_size:', bs)
            print('Learning_rate:', lr)
            
            # try: 
            model = torch.load('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)).to(device)
            #model = torch.load('./Model/%s/%s_1_%s_%s.pt'%(file_nam ,model_nam, target, info)).to(device)
            # sel_time -> LABEL
            if os.path.exists('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)):
                model_size_mb = os.path.getsize('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)) / (1024 * 1024)
                print(f"📦 Model size: {model_size_mb:.2f} MB → {os.path.basename('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr))}")
            else:
                print(f"⚠️ There is no model: {'./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)}")

            model.eval()
            y_pred_te = model(pred_dataset.x)
            y_pred_te = y_pred_te.detach().cpu().numpy() * (max_[0] - min_[0]) + min_[0]

            p_dat = pd.concat([y_te, pd.DataFrame(y_pred_te, index = sel_time[:])], axis = 1)
            fcts = p_dat.iloc[-1, :]

            for i in range(7):
                p_dat.iloc[:, i+1] = p_dat.iloc[:, i+1].shift(i+1)

            p_dat['mean'] = p_dat.iloc[:, 1:7].mean(axis = 1)
            p_dat['std'] = p_dat.iloc[:, 1:7].std(axis = 1)

            p_dat_ = p_dat.dropna()
            p_dat.to_csv(f"./Results/{file_nam}/{model_nam}_predictions_{target}.csv")

            start_time = p_dat.index[-1] + pd.Timedelta(days = 1)
            end_time = start_time + pd.Timedelta(days = 6)

            f_dat = pd.DataFrame(np.array(fcts[1:]).reshape(-1, 1), columns = ['forecasts'],
                        index = pd.date_range(start_time, end_time, freq = "D"))
            f_dat.to_csv(f"./Results/{file_nam}/{model_nam}_forcasts_{target}.csv")

            p_dat_total = pd.concat([p_dat_total, p_dat], axis = 1)
            f_dat_total = pd.concat([f_dat_total, f_dat], axis = 1)
            ## model = torch.load(###)

            p_dat_ = np.array(p_dat.dropna())

            quantile_loss, mse_loss  = quantileloss(p_dat_, min_, max_, quantile = 0.8)
            #print(quantile_loss)
            import numpy as np
            import matplotlib.pyplot as plt
            import scipy.stats as stats

            # Example data for p_dat, min_, and max_
            min = min_[0]
            max = max_[0]
            p_dat2 = (p_dat_ - min)/(max - min)


            # Combine all errors for QQ plot
            all_errors = []

            # Loop through the 7 predictions
            for i in range(7):
                errors = p_dat2[:, 0] - p_dat2[:, i + 1]  # Error calculation
                all_errors.extend(errors)  # Collect all errors for QQ plot

            # Convert to numpy array
            all_errors = np.array(all_errors)

            print(quantile_loss, mse_loss)
            for i in range(7):
                print(i+1 ,'일 후 예측결과')
                perf_save_temp = performance_tab_mod(p_dat_[:, 0], p_dat_[:,i+1],
                                                i+1,
                                                model_nam,
                                                info = seq_len, 
                                                target = target)

                rrmse_, nnse_, nse_, rmse, mse, mae, rmse_, mse_, mae_, rmse_sum, mse_sum, mae_sum = loss_summertime(p_dat, min_, max_, i)
                perf_save_temp['hid'] = hidden_dim
                perf_save_temp['num_layers'] = num_layers
                perf_save_temp['batch_size'] = bs
                perf_save_temp['learning rate'] = lr
                perf_save_temp['file_nam'] = file_nam                               

                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rrmse'] = rrmse_
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nnse'] = nnse_
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nse'] = nse_        
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse'] = mse
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae'] = mae
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_'] = rmse_
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_'] = mse_
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_'] = mae_
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_sum'] = rmse_sum
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_sum'] = mse_sum
                perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_sum'] = mae_sum

                performance_table = pd.concat([performance_table, perf_save_temp], axis = 0)

            #     confusion_table = pd.concat([confusion_table, confusion_temp], axis = 0)
                
                if i in [0,6]:

                    #print(i+1, '일 후 R2 :', perf_save_temp['r2'].astype(float).values[0])
                    plt.figure(figsize=(10, 4))
                    #plt.axhline(y=10, color='gray', linestyle='--', linewidth=1, alpha=0.5)
                    #plt.axhline(y=30, color='gray', linestyle='--', linewidth=1, alpha=0.5)
                    #plt.axhline(y=100, color='gray', linestyle='--', linewidth=1, alpha=0.5)

                    plt.plot(p_dat[target], c = 'blue', linewidth = 1)
                    plt.plot(p_dat.iloc[:, i+1], c = 'red', linewidth = 1)
                    #plt.plot(p_dat_mod.iloc[:, i+1], c = 'red', linewidth = 1)
                    #plt.plot(f_dat['forecasts'], color = 'red', linewidth = 1)
                    
                    #plt.title(target)
                    plt.xlim(p_dat.index[0], p_dat.index[-1])
                    plt.ylim((0, 400))
                    plt.yticks(size=10)
                    plt.ylabel('Predicted Turbidity (NTU)', fontsize=10)
                    plt.tight_layout()
                    #plt.savefig(f'./Results/{file_nam}/{model_nam}_{target}_1_{seq_len}_hid_{hidden_dim}_num_{num_layers}_bs_{bs}_lr_{lr}_temporal_variation.png', 
                    #        dpi=256, transparent=True)

                    #if perf_save_temp['r2'].astype(float).values[0] >= 0:
                    plt.show()
                    #else:
                    #    plt.close()
                print("")
            # except:
            #     print('No exist model')

            # confusion_result = confusion_matrix_file(confusion_table)
            # confusion_result.to_csv(f"./Results/{file_nam}/confusion_{file_nam}.csv")
        # performance_table['target'] = 'NTU'
        # performance_table['site'] = performance_table['site'].apply(lambda x: x.split('.')[0])
        # performance_table = performance_table[['site', 'target', 'model', 
        #             'fct', 'accuracy', 'recall', 'r2', 'rmse', 'rrmse',
        #             'nnse', 'nse', 'mse', 'mae', 'rmse_', 'mse_', 'mae_', 'rmse_sum',
        #     'mse_sum', 'mae_sum']]
        #performance_table.to_csv(f"./Performance_{file_nam}.csv")

## Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)  # Add batch dimension

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :].to(x.device)

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, nhead=8):
        super(TransformerEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=nhead)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(hidden_dim)

    def forward(self, input):
        input = self.input_projection(input)  # Project input to hidden_dim size
        input = self.positional_encoding(input)  # Apply positional encoding
        output = self.encoder(input.transpose(0, 1))  # Transformer expects seq_len first
        return output.transpose(0, 1)  # Return batch-first format

class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim, num_layers, output_dim, nhead=8):
        super(TransformerDecoder, self).__init__()
        self.output_projection = nn.Linear(hidden_dim, output_dim)
        decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=nhead)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(hidden_dim)

    def forward(self, input, encoder_output):
        input = self.positional_encoding(input)  # Apply positional encoding
        output = self.decoder(input.transpose(0, 1), encoder_output.transpose(0, 1))
        output = output.transpose(0, 1)
        return self.output_projection(output)


class TransformerSeq2Seq(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, forecast_horizon, nhead=8):
        super(TransformerSeq2Seq, self).__init__()
        self.encoder = TransformerEncoder(input_dim, hidden_dim, num_layers, nhead=nhead)
        self.decoder = TransformerDecoder(hidden_dim, num_layers, output_dim, nhead=nhead)
        self.forecast_horizon = forecast_horizon

    def forward(self, input, known_features):
        out_seq_len = self.forecast_horizon
        encoder_output = self.encoder(input)
        
        # 첫 번째 디코더 입력값으로 인코더 마지막 출력을 사용
        decoder_input = encoder_output[:, -1:, :]  # (batch_size, 1, hidden_dim)
        outputs = []

        for t in range(out_seq_len):
            # 디코더에서 예측된 현재 시점의 출력
            decoder_output = self.decoder(decoder_input, encoder_output)
            current_output = decoder_output[:, -1, :]  # (batch_size, hidden_dim)
            outputs.append(current_output)
            
            # 새로운 예측값을 디코더 입력값으로 설정
            decoder_input = current_output.unsqueeze(1)  # (batch_size, 1, hidden_dim)
        
        # 모든 시점의 예측값을 시퀀스 형태로 변환
        outputs = torch.stack(outputs, dim=1)  # (batch_size, out_seq_len, output_dim)
        return outputs

In [ ]:
dep_vars = ['PD1.NTU', 'PD2.NTU', 'PD3.NTU']
predictors = ['CJdam.Prec', 'PDdam.Prec', 'CJdam.DIS', 'CPdam.DIS', 'PDdam.DIS',
            #   'YP.NTU', 'GP.NTU', 
              'CP.Prec', 'GJ.Prec', 'YP.Prec', 'CJdam.NTU', 'SYdam.NTU'
            #   'YP.WTemp', 'GP.WTemp'
              ]
known_vars= []
Variables = dep_vars + predictors

file_nam = 'Turb_Transformer'
loss_nam = 'huber_loss'

In [ ]:
import warnings
warnings.filterwarnings('ignore')
sepa = "2023-01-01"
model_nam = "Transformer"
info = '20241008'
final_performance = pd.DataFrame()
seq_len = 36
fct_h = 1
h_len = 7

#### Perform imputation
from SaaS_WQ.preprocessing import imputation_KF
from SaaS_WQ.plot import matrix

performance_table = pd.DataFrame()
for target in dep_vars:

    if target == 'PD1.NTU':
        hyperparameter_combinations = [(32, 3, 128, 0.005)]
    elif target == 'PD2.NTU':
        hyperparameter_combinations = [(16, 5, 64, 0.0005)]
    elif target == 'PD3.NTU':
        hyperparameter_combinations = [(32, 5, 128, 0.0001)]

    Model_future = pd.read_csv(f'./Data/Model_known_.csv')
    test_data_imp = pd.read_csv(f"./Data/test_data_sensor_.csv", index_col=0, encoding = "cp949")
    test_data_imp.index = pd.to_datetime(test_data_imp.index)
    test_data_imp = test_data_imp[test_data_imp.index >= pd.to_datetime(sepa)]

    # ###########
    # test_data_imp = test_data_imp[test_data_imp.index<='2024-07-19 10:00']

    Model_future['Date'] = pd.to_datetime(Model_future['Date'])
    Model_future['Date'] = Model_future['Date'] + pd.Timedelta(days = 2) 

    test_knw = Model_future[Model_future['Date'] >= pd.to_datetime(sepa)]

    ##### For create dataset
    class TimeSeriesDataset_pred(Dataset):
        def __init__(self, x, knw, device):
            self.x = torch.tensor(x, dtype=torch.float32).to(device)
            self.knw = torch.tensor(knw, dtype=torch.float32).to(device)
            
        def __len__(self):
            return len(self.x)
        
        def __getitem__(self, index):
            return self.x[index], self.knw[index]
        
    #performance_table = pd.DataFrame()
    confusion_table = pd.DataFrame()


    wrk_t = 9
    test_data_ = test_data_imp.loc[:, [target] + predictors]

    te_x_ = []
    te_knw_ = []

    for t in test_data_[test_data_.index.hour == wrk_t].index:
        end_time = t
        start_time = end_time - pd.Timedelta(hours = seq_len - 1)
        
        x = test_data_.loc[start_time:end_time, predictors].values
        if x.shape[0] == seq_len:
            te_x_.append(x)
        
    te_x_ = np.array(te_x_)

    min_, max_ = pickle.load(open(f'./Model/{file_nam}/{target}_Minmax.pkl', 'rb'))

    #### For target sequence
    sel_time = test_data_[test_data_.index.hour == wrk_t][-te_x_.shape[0]:].index.floor('D')
    for t in sel_time[:-1]:
        start_time = t + pd.Timedelta(days = fct_h) 

        knw = test_knw.loc[pd.to_datetime(test_knw['Date']) == t, known_vars].values
        te_knw_.append(knw)
            
    #te_knw_ = np.array(te_knw_)/100

    #te_x_ = te_x_[:-1, :, :]

    # print(te_x_.shape, te_knw_.shape)

    te_x__ = (te_x_ - min_[1:])/(max_[1:] - min_[1:])

    pred_dataset = TimeSeriesDataset_pred(te_x__, te_knw_, device)

    test_data_.loc[:, 'date'] = test_data_.index.date
    test_data__ = test_data_.groupby('date')[target].mean().values
    y_te = pd.DataFrame(test_data__, index = test_data_.index.floor('D').unique(), columns = [target])
    y_te = y_te.loc[sel_time, :]

    for hidden_dim, num_layers, bs, lr in hyperparameter_combinations:

        print(model_nam)
        print(target)
        print('hidden_dim:', hidden_dim)
        print('num_layer:', num_layers)
        print('batch_size:', bs)
        print('Learning_rate:', lr)
        
        # try:
        model = torch.load('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt' % 
                (file_nam, loss_nam,model_nam, target, bs, hidden_dim, num_layers, lr)).to(device)
        #model = torch.load('./Model/%s/%s_1_%s_%s.pt'%(file_nam ,model_nam, target, info)).to(device)
        # sel_time -> LABEL
        if os.path.exists('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)):
            model_size_mb = os.path.getsize('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)) / (1024 * 1024)
            print(f"📦 Model size: {model_size_mb:.2f} MB → {os.path.basename('./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr))}")
        else:
            print(f"⚠️ There is no model: {'./Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt'%(file_nam, loss_nam, model_nam, target, bs, hidden_dim, num_layers, lr)}")
                        # sel_time -> LABEL``
        model.eval()
        y_pred_te = model(pred_dataset.x, pred_dataset.knw)
        y_pred_te = y_pred_te.detach().cpu().numpy() * (max_[0] - min_[0]) + min_[0]
        y_pred_te = y_pred_te.squeeze(-1)

        p_dat = pd.concat([y_te, pd.DataFrame(y_pred_te, index = sel_time)], axis = 1)
        fcts = p_dat.iloc[-1, :]

        for i in range(7):
            p_dat.iloc[:, i+1] = p_dat.iloc[:, i+1].shift(i+1)

        p_dat['mean'] = p_dat.iloc[:, 1:7].mean(axis = 1)
        p_dat['std'] = p_dat.iloc[:, 1:7].std(axis = 1)

        p_dat_ = p_dat.dropna()
        #p_dat.to_csv(f"./Results/{file_nam}/predictions_{target}.csv")

        start_time = p_dat.index[-1] + pd.Timedelta(days = 1)
        end_time = start_time + pd.Timedelta(days = 6)

        f_dat = pd.DataFrame(np.array(fcts[1:]).reshape(-1, 1), columns = ['forecasts'],
                    index = pd.date_range(start_time, end_time, freq = "D"))
        #f_dat.to_csv(f"./Results/{file_nam}/forcasts_{target}.csv")

        ## model = torch.load(###)

        p_dat_ = np.array(p_dat.dropna())
        quantile_loss, mse_loss  = quantileloss(p_dat_, min_, max_, quantile = 0.8)
        # if not os.path.exists('./Results/%s'%(file_nam)):
        #     os.mkdir('./Results/%s'%(file_nam))
        print(quantile_loss)
        for i in range(7):
            print(target, i+1 ,'일 후 예측결과', np.round(f_dat.iloc[i, 0], 3))
            perf_save_temp = performance_tab_mod(p_dat_[:, 0], p_dat_[:,i+1],
                                            i+1,
                                            model_nam,
                                            info = seq_len, 
                                            target = target)
            
            confusion_temp = confusion_tab_pred(p_dat_[:, 0], p_dat_[:,i+1],
                                            i+1,
                                            model_nam,
                                            info = seq_len, 
                                            target = target)

            rrmse_, nnse_, nse_, rmse, mse, mae, rmse_, mse_, mae_, rmse_sum, mse_sum, mae_sum = loss_summertime(p_dat, min_, max_, i)
            perf_save_temp['hid'] = hidden_dim
            perf_save_temp['num_layers'] = num_layers
            perf_save_temp['batch_size'] = bs
            perf_save_temp['learning rate'] = lr                            

            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rrmse'] = rrmse_
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nnse'] = nnse_
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'nse'] = nse_        
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse'] = mse
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae'] = mae
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_'] = rmse_
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_'] = mse_
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_'] = mae_
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'rmse_sum'] = rmse_sum
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mse_sum'] = mse_sum
            perf_save_temp.loc[(perf_save_temp['site'] == target) & (perf_save_temp['fct'] == i+1) & (perf_save_temp['model'] == model_nam), 'mae_sum'] = mae_sum

            performance_table = pd.concat([performance_table, perf_save_temp], axis = 0)
            confusion_table = pd.concat([confusion_table, confusion_temp], axis = 0)

            confusion_result = confusion_matrix_file(confusion_table)
            
            if i in [0,6]:

                #print(i+1, '일 후 R2 :', perf_save_temp['r2'].astype(float).values[0])
                plt.figure(figsize=(10, 4))
                #plt.axhline(y=10, color='gray', linestyle='--', linewidth=1, alpha=0.5)
                #plt.axhline(y=30, color='gray', linestyle='--', linewidth=1, alpha=0.5)
                #plt.axhline(y=100, color='gray', linestyle='--', linewidth=1, alpha=0.5)

                plt.plot(p_dat[target], c = 'blue', linewidth = 1)
                plt.plot(p_dat.iloc[:, i+1], c = 'red', linewidth = 1)
                #plt.plot(p_dat_mod.iloc[:, i+1], c = 'red', linewidth = 1)
                #plt.plot(f_dat['forecasts'], color = 'red', linewidth = 1)
                
                #plt.title(target)
                plt.xlim(p_dat.index[0], p_dat.index[-1])
                plt.ylim((0, 400))
                plt.yticks(size=10)
                plt.ylabel('Predicted Turbidity (NTU)', fontsize=10)
                plt.tight_layout()
                #plt.savefig(f'./Results/{file_nam}/{model_nam}_{target}_1_{seq_len}_hid_{hidden_dim}_num_{num_layers}_bs_{bs}_lr_{lr}_temporal_variation.png', 
                #        dpi=256, transparent=True)

                #if perf_save_temp['r2'].astype(float).values[0] >= 0:
                plt.show()
                #else:
                #    plt.close()
            print("")
        # except:
        #     print('No Model')

# performance_table['target'] = 'NTU'
# performance_table['site'] = performance_table['site'].apply(lambda x: x.split('.')[0])
# performance_table = performance_table[['site', 'target', 'model', 
#             'fct', 'accuracy', 'recall', 'r2', 'rmse', 'rrmse',
#             'nnse', 'nse', 'mse', 'mae', 'rmse_', 'mse_', 'mae_', 'rmse_sum',
#     'mse_sum', 'mae_sum']]
    #confusion_result = confusion_matrix_file(confusion_table)
    #confusion_result.to_csv(f"./Results/{file_nam}/confusion_{file_nam}.csv")
    #performance_table.to_csv(f"./Results/{file_nam}/performance_{file_nam}.csv")
    #final_performance = pd.concat([final_performance,performance_table], axis = 0)
    #final_performance.to_csv("./Results/performance_Transformer.csv")